In [1]:
import os
import sys
import platform
import subprocess
from pathlib import Path

print("===== Python環境 =====")
print("Python:", sys.version)
print("OS:", platform.platform())
print("Current directory:", os.getcwd())

print("\n===== コマンド環境 =====")
commands = [
    ["node", "--version"],
    ["npm", "--version"],
    ["git", "--version"],
]

for command in commands:
    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            check=True,
        )
        print(f"{command[0]}:", result.stdout.strip())
    except Exception as error:
        print(f"{command[0]}: 使用不可", repr(error))

print("\n===== PyTorch環境 =====")
try:
    import torch

    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version:", torch.version.cuda)

    if torch.cuda.is_available():
        print("GPU count:", torch.cuda.device_count())
        print("GPU name:", torch.cuda.get_device_name(0))
    else:
        print("GPU: 現在は使用されていません")
except Exception as error:
    print("PyTorch読み込み失敗:", repr(error))

print("\n===== Kaggleディレクトリ =====")
print("/kaggle/input exists:", Path("/kaggle/input").exists())
print("/kaggle/working exists:", Path("/kaggle/working").exists())

working_test = Path("/kaggle/working/tactical_hub_test.txt")

try:
    working_test.write_text(
        "Kaggle write test successful",
        encoding="utf-8",
    )
    print("/kaggle/working write: 成功")
    working_test.unlink()
except Exception as error:
    print("/kaggle/working write: 失敗", repr(error))

print("\n===== 判定 =====")

python_ok = sys.version_info >= (3, 10)

try:
    node_major = int(
        subprocess.run(
            ["node", "--version"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip().lstrip("v").split(".")[0]
    )
    node_ok = node_major >= 18
except Exception:
    node_ok = False

try:
    import torch
    torch_ok = True
except Exception:
    torch_ok = False

print("Python 3.10以上:", python_ok)
print("Node.js 18以上:", node_ok)
print("PyTorch使用可能:", torch_ok)

if python_ok and node_ok and torch_ok:
    print("\n基本環境は合格です。")
else:
    print("\n不足している環境があります。上の結果を確認してください。")

===== Python環境 =====
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
OS: Linux-6.12.90+-x86_64-with-glibc2.35
Current directory: /kaggle/working

===== コマンド環境 =====
node: v20.19.0
npm: 10.8.2
git: git version 2.34.1

===== PyTorch環境 =====
PyTorch: 2.10.0+cpu
CUDA available: False
CUDA version: None
GPU: 現在は使用されていません

===== Kaggleディレクトリ =====
/kaggle/input exists: True
/kaggle/working exists: True
/kaggle/working write: 成功

===== 判定 =====
Python 3.10以上: True
Node.js 18以上: True
PyTorch使用可能: True

基本環境は合格です。


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("GPUを認識できていません")

PyTorch: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU count: 2
GPU name: Tesla T4


In [2]:
from pathlib import Path
import os
import subprocess

repo_url = "https://github.com/YoshimatsuKeisei/Tactical-hub.git"
repo_dir = Path("/kaggle/working/Tactical-hub")

print("===== Tactical-hub取得 =====")

if repo_dir.exists():
    print("既に存在します:", repo_dir)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", repo_url, str(repo_dir)],
        check=True,
    )
    print("clone成功:", repo_dir)

os.chdir(repo_dir)

print("\n===== Git情報 =====")
subprocess.run(["git", "branch", "--show-current"], check=True)
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)

print("\n===== 主要ファイル確認 =====")
for name in [
    "package.json",
    "package-lock.json",
    "requirements.txt",
    "pyproject.toml",
]:
    path = repo_dir / name
    print(f"{name}: {'あり' if path.exists() else 'なし'}")

print("\nCurrent directory:", os.getcwd())
print("ソースコード取得完了")

===== Tactical-hub取得 =====


Cloning into '/kaggle/working/Tactical-hub'...


clone成功: /kaggle/working/Tactical-hub

===== Git情報 =====
main
42bc4f7

===== 主要ファイル確認 =====
package.json: なし
package-lock.json: なし
requirements.txt: なし
pyproject.toml: なし

Current directory: /kaggle/working/Tactical-hub
ソースコード取得完了


In [1]:
from pathlib import Path
import subprocess
import json
import shutil
import os

repo_url = "https://github.com/YoshimatsuKeisei/Tactical-hub.git"
repo_dir = Path("/kaggle/working/Tactical-hub")

print("===== 現在の作業領域 =====")
print("Current directory:", os.getcwd())
print("/kaggle/working:", list(Path("/kaggle/working").iterdir()))

print("\n===== Tactical-hub取得 =====")

if repo_dir.exists():
    print("既に存在します:", repo_dir)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", repo_url, str(repo_dir)],
        check=True,
    )
    print("clone成功:", repo_dir)

print("\n===== リポジトリ直下 =====")
for path in sorted(repo_dir.iterdir()):
    kind = "DIR " if path.is_dir() else "FILE"
    print(f"{kind}: {path.name}")

print("\n===== package.json探索 =====")

package_files = [
    path
    for path in repo_dir.rglob("package.json")
    if "node_modules" not in path.parts
]

if not package_files:
    print("package.jsonは見つかりませんでした。")
else:
    for package_path in package_files:
        print(f"\n発見: {package_path}")

        try:
            data = json.loads(package_path.read_text(encoding="utf-8"))
            print("name:", data.get("name", "未設定"))

            scripts = data.get("scripts", {})
            related_scripts = {
                name: command
                for name, command in scripts.items()
                if any(
                    keyword in name.lower()
                    for keyword in ["rl", "bc", "train"]
                )
            }

            if related_scripts:
                print("学習関連スクリプト:")
                for name, command in related_scripts.items():
                    print(f"  {name}: {command}")
            else:
                print("学習関連スクリプトなし")

        except Exception as error:
            print("読み込み失敗:", repr(error))

===== 現在の作業領域 =====
Current directory: /kaggle/working
/kaggle/working: [PosixPath('/kaggle/working/.virtual_documents')]

===== Tactical-hub取得 =====


Cloning into '/kaggle/working/Tactical-hub'...


clone成功: /kaggle/working/Tactical-hub

===== リポジトリ直下 =====
DIR : .VSCodeCounter
DIR : .git
DIR : Tactical-hub
FILE: domain_model_v0.1.md
FILE: map_spec_v0.1.md
FILE: rule_spec_v0.1.md
FILE: screen_transition_diagrams_v0.1.md
FILE: sequence_diagrams_simple_v0.2.md
FILE: state_transition_diagrams_v0.1.md

===== package.json探索 =====

発見: /kaggle/working/Tactical-hub/Tactical-hub/package.json
name: 未設定
学習関連スクリプト:
  rl:smoke: vite-node src/game/cpu/rlSmokeCli.ts
  rl:collect: vite-node src/game/cpu/rlImitationCli.ts
  rl:replay: vite-node src/game/cpu/rlImitationReplayCli.ts
  rl:prepare-sidecars: vite-node src/game/cpu/rlPrepareSidecarsCli.ts
  rl:bc: vite-node src/game/cpu/rlBehavioralCloningCli.ts
  rl:bc-profile: vite-node src/game/cpu/rlBcProfilerCli.ts


In [2]:
from pathlib import Path
import os
import subprocess

project_dir = Path("/kaggle/working/Tactical-hub/Tactical-hub")

print("===== プロジェクト確認 =====")
print("Project directory:", project_dir)
print("Exists:", project_dir.exists())

required_files = [
    "package.json",
    "package-lock.json",
]

for filename in required_files:
    path = project_dir / filename
    print(f"{filename}: {'あり' if path.exists() else 'なし'}")

if not project_dir.exists():
    raise FileNotFoundError(
        "プロジェクトがありません。セッション切替後なら再度git cloneが必要です。"
    )

if not (project_dir / "package.json").exists():
    raise FileNotFoundError("package.jsonが見つかりません。")

os.chdir(project_dir)
print("Current directory:", os.getcwd())

print("\n===== npm依存関係のインストール =====")
subprocess.run(
    [
        "npm",
        "ci",
        "--no-audit",
        "--no-fund",
    ],
    check=True,
)

print("\n===== インストール後の確認 =====")
subprocess.run(["node", "--version"], check=True)
subprocess.run(["npm", "--version"], check=True)
subprocess.run(
    ["npx", "vite-node", "--version"],
    check=True,
)

print("\nnode_modules:", (project_dir / "node_modules").exists())

print("\n===== Python設定ファイル探索 =====")
python_config_names = {
    "requirements.txt",
    "requirements-dev.txt",
    "pyproject.toml",
    "environment.yml",
}

found = []

for path in project_dir.rglob("*"):
    if path.is_file() and path.name in python_config_names:
        if "node_modules" not in path.parts:
            found.append(path)

if found:
    for path in found:
        print(path)
else:
    print("Python依存関係ファイルは見つかりませんでした。")

print("\n依存関係のインストール完了")

===== プロジェクト確認 =====
Project directory: /kaggle/working/Tactical-hub/Tactical-hub
Exists: True
package.json: あり
package-lock.json: あり
Current directory: /kaggle/working/Tactical-hub/Tactical-hub

===== npm依存関係のインストール =====


npm warn deprecated glob@10.5.0: Old versions of glob are not supported, and contain widely publicized security vulnerabilities, which have been fixed in the current version. Please update. Support for old versions may be purchased (at exorbitant rates) by contacting i@izs.me



added 167 packages in 5s

===== インストール後の確認 =====
v20.19.0
10.8.2


npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.2
npm notice To update run: npm install -g npm@12.0.2
npm notice


vite-node/2.1.8 linux-x64 node-v20.19.0

node_modules: True

===== Python設定ファイル探索 =====
Python依存関係ファイルは見つかりませんでした。

依存関係のインストール完了


In [5]:
import os
import subprocess
from pathlib import Path

project_dir = Path("/kaggle/working/Tactical-hub/Tactical-hub")
os.chdir(project_dir)

print("===== Kaggle短縮スモークテスト =====")

result = subprocess.run(
    [
        "npm",
        "run",
        "rl:smoke",
        "--",
        "--max-decisions",
        "100",
        "--device",
        "cpu",
    ],
    text=True,
    timeout=180,
)

print("\n===== 終了結果 =====")
print("return code:", result.returncode)

===== Kaggle短縮スモークテスト =====

> rl:smoke
> vite-node src/game/cpu/rlSmokeCli.ts --max-decisions 100 --device cpu



The CJS build of Vite's Node API is deprecated. See https://vite.dev/guide/troubleshooting.html#vite-cjs-node-api-deprecated for more details.
[RL Device] requested device=cpu | selected device=cpu | PyTorch=2.10.0+cpu | cuda.is_available=False


{
  "seed": 1,
  "decisionCount": 100,
  "terminal": false,
  "turnNumber": 4,
  "teams": [
    {
      "teamId": "team-1",
      "status": "active",
      "unitCount": 3,
      "baseCount": 1
    },
    {
      "teamId": "team-2",
      "status": "active",
      "unitCount": 3,
      "baseCount": 1
    },
    {
      "teamId": "team-3",
      "status": "active",
      "unitCount": 3,
      "baseCount": 1
    },
    {
      "teamId": "team-4",
      "status": "active",
      "unitCount": 3,
      "baseCount": 1
    }
  ],
  "loserTeamIds": [],
  "endReason": "decision_limit",
  "pythonAbnormalExit": false,
  "selectedDevice": "cpu"
}

===== 終了結果 =====
return code: 1


In [9]:
from pathlib import Path

input_root = Path("/kaggle/input")

print("===== Kaggle Input一覧 =====")

for path in sorted(input_root.iterdir()):
    print(
        "DIR " if path.is_dir() else "FILE",
        path,
    )

print("\n===== BCデータ探索 =====")

target_names = {
    "bc100",
    "bc100.rng-sidecars",
}

found = []

for path in input_root.rglob("*"):
    if path.name in target_names:
        found.append(path)
        print(
            f"{path.name}:",
            path,
            "| directory:", path.is_dir(),
        )

print("\n===== ZIPファイル探索 =====")

zip_files = list(input_root.rglob("*.zip"))

if zip_files:
    for path in zip_files:
        print(path)
else:
    print("ZIPファイルなし")

print("\n===== 判定 =====")

found_names = {path.name for path in found}

if target_names.issubset(found_names):
    print("bc100とsidecarの両方を確認できました。")
else:
    print("必要な2フォルダがまだ揃っていません。")

===== Kaggle Input一覧 =====
DIR  /kaggle/input/datasets

===== BCデータ探索 =====
bc100.rng-sidecars: /kaggle/input/datasets/keisabo/imitation-learning-dataset/bc100.rng-sidecars | directory: True
bc100: /kaggle/input/datasets/keisabo/imitation-learning-dataset/bc100 | directory: False

===== ZIPファイル探索 =====
ZIPファイルなし

===== 判定 =====
bc100とsidecarの両方を確認できました。


In [1]:
from pathlib import Path
import os
import subprocess
import time

project_dir = Path("/kaggle/working/Tactical-hub/Tactical-hub")

data_path = Path(
    "/kaggle/input/datasets/keisabo/imitation-learning-dataset/bc100"
)

sidecar_path = Path(
    "/kaggle/input/datasets/keisabo/imitation-learning-dataset/"
    "bc100.rng-sidecars"
)

checkpoint_path = Path(
    "/kaggle/working/rl-checkpoints/bc-smoke.pt"
)

print("===== 実行前確認 =====")
print("プロジェクト:", project_dir.exists())
print("教師データ:", data_path.is_file())
print("sidecar数:", len(list(sidecar_path.glob("*.json"))))

if not project_dir.exists():
    raise FileNotFoundError("プロジェクトが見つかりません。")

if not data_path.is_file():
    raise FileNotFoundError("bc100が見つかりません。")

if len(list(sidecar_path.glob("*.json"))) != 100:
    raise RuntimeError("sidecarが100個確認できません。")

checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

# 古いテスト結果があれば削除
if checkpoint_path.exists():
    checkpoint_path.unlink()

os.chdir(project_dir)

command = [
    "npm",
    "run",
    "rl:bc",
    "--",
    "--data",
    str(data_path),
    "--epochs",
    "1",
    "--batch-size",
    "8",
    "--learning-rate",
    "0.0001",
    "--checkpoint",
    str(checkpoint_path),
    "--train-range",
    "1-1",
    "--validation-range",
    "2-2",
    "--test-range",
    "3-3",
    "--seed",
    "1",
    "--workers",
    "1",
    "--torch-threads",
    "2",
    "--torch-interop-threads",
    "1",
    "--device",
    "cpu",
]

print("\n===== 模倣学習テスト開始 =====")
print(" ".join(command))
print()

started = time.time()

result = subprocess.run(
    command,
    text=True,
    check=False,
)

elapsed = time.time() - started

print("\n===== 実行後確認 =====")
print("return code:", result.returncode)
print(f"経過秒数: {elapsed:.1f}")
print("チェックポイント存在:", checkpoint_path.is_file())

if checkpoint_path.is_file():
    print(
        "チェックポイント容量:",
        f"{checkpoint_path.stat().st_size / 1024 / 1024:.2f} MB",
    )

if result.returncode == 0 and checkpoint_path.is_file():
    print("\n模倣学習テスト成功です。")
else:
    print("\n模倣学習テストに問題があります。")

===== 実行前確認 =====
プロジェクト: False
教師データ: True
sidecar数: 100


FileNotFoundError: プロジェクトが見つかりません。

In [2]:
from pathlib import Path
import subprocess
import os
import shutil

repo_root = Path("/kaggle/working/Tactical-hub")
project_dir = repo_root / "Tactical-hub"

print("===== プロジェクト復元 =====")

if not project_dir.exists():
    if repo_root.exists():
        shutil.rmtree(repo_root)

    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/YoshimatsuKeisei/Tactical-hub.git",
            str(repo_root),
        ],
        check=True,
    )
    print("GitHubから取得しました。")
else:
    print("プロジェクトは既に存在します。")

if not (project_dir / "package.json").is_file():
    raise FileNotFoundError(
        f"package.jsonが見つかりません: {project_dir}"
    )

os.chdir(project_dir)

print("\n===== npm依存関係の復元 =====")

if not (project_dir / "node_modules").exists():
    subprocess.run(
        [
            "npm",
            "ci",
            "--no-audit",
            "--no-fund",
        ],
        check=True,
    )
    print("npm依存関係をインストールしました。")
else:
    print("node_modulesは既に存在します。")

print("\n===== 最終確認 =====")
print("プロジェクト:", project_dir.exists())
print("package.json:", (project_dir / "package.json").is_file())
print("node_modules:", (project_dir / "node_modules").is_dir())

print("\n復元完了です。")

===== プロジェクト復元 =====


Cloning into '/kaggle/working/Tactical-hub'...


GitHubから取得しました。

===== npm依存関係の復元 =====


npm warn deprecated glob@10.5.0: Old versions of glob are not supported, and contain widely publicized security vulnerabilities, which have been fixed in the current version. Please update. Support for old versions may be purchased (at exorbitant rates) by contacting i@izs.me



added 167 packages in 6s
npm依存関係をインストールしました。

===== 最終確認 =====
プロジェクト: True
package.json: True
node_modules: True

復元完了です。


npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.2
npm notice To update run: npm install -g npm@12.0.2
npm notice


># 復帰用セル セッションが切断されたりしたときにgithubのコードなどを復活させる 

In [3]:
# 復帰用セル

from pathlib import Path
import subprocess
import shutil
import hashlib
import os

# =========================================================
# 固定設定
# =========================================================

REPO_URL = "https://github.com/YoshimatsuKeisei/Tactical-hub.git"
EXPECTED_COMMIT = "1ef9943f0b635f8fbfdf1c2f9a6c91a2e5adc6f3"

REPO_ROOT = Path("/kaggle/working/Tactical-hub")
PROJECT_DIR = REPO_ROOT / "Tactical-hub"

DATASET_ROOT = Path(
    "/kaggle/input/datasets/keisabo/imitation-learning-dataset"
)
DATA_PATH = DATASET_ROOT / "bc100"
SIDECAR_DIR = DATASET_ROOT / "bc100.rng-sidecars"

OUTPUT_ROOT = Path("/kaggle/working/tactical-hub-output")
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
LOG_DIR = OUTPUT_ROOT / "logs"

print("========================================")
print(" Tactical-hub Kaggle復帰処理")
print("========================================")

# =========================================================
# 1. ソースコード復元
# =========================================================

print("\n===== 1. ソースコード =====")

if not (REPO_ROOT / ".git").is_dir():
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)

    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            "main",
            "--depth",
            "20",
            REPO_URL,
            str(REPO_ROOT),
        ],
        check=True,
    )

    print("GitHubからcloneしました。")
else:
    print("既存のリポジトリを使用します。")

# mainの最新状態を取得
subprocess.run(
    [
        "git",
        "-C",
        str(REPO_ROOT),
        "fetch",
        "--depth",
        "20",
        "origin",
        "main",
    ],
    check=True,
)

# Kaggle実地試験では今回の改修コミットを固定使用
subprocess.run(
    [
        "git",
        "-C",
        str(REPO_ROOT),
        "checkout",
        "--detach",
        EXPECTED_COMMIT,
    ],
    check=True,
)

current_commit = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
    text=True,
).strip()

print("取得コミット:", current_commit)

if current_commit != EXPECTED_COMMIT:
    raise RuntimeError(
        f"想定コミットと一致しません: {current_commit}"
    )

package_json = PROJECT_DIR / "package.json"
package_lock = PROJECT_DIR / "package-lock.json"

if not package_json.is_file():
    raise FileNotFoundError(
        f"package.jsonが見つかりません: {package_json}"
    )

# =========================================================
# 2. npm依存関係復元
# =========================================================

print("\n===== 2. npm依存関係 =====")

node_modules = PROJECT_DIR / "node_modules"
lock_marker = PROJECT_DIR / ".kaggle-package-lock.sha256"

lock_hash = hashlib.sha256(
    package_lock.read_bytes()
).hexdigest()

saved_hash = (
    lock_marker.read_text(encoding="utf-8").strip()
    if lock_marker.is_file()
    else ""
)

if not node_modules.is_dir() or saved_hash != lock_hash:
    os.chdir(PROJECT_DIR)

    subprocess.run(
        [
            "npm",
            "ci",
            "--no-audit",
            "--no-fund",
        ],
        check=True,
    )

    lock_marker.write_text(
        lock_hash,
        encoding="utf-8",
    )

    print("npm ciを実行しました。")
else:
    print("既存のnode_modulesを使用します。")

# =========================================================
# 3. 教師データ確認
# =========================================================

print("\n===== 3. 教師データ =====")

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"教師データがありません: {DATA_PATH}"
    )

if not SIDECAR_DIR.is_dir():
    raise FileNotFoundError(
        f"sidecarフォルダがありません: {SIDECAR_DIR}"
    )

sidecar_count = len(list(SIDECAR_DIR.glob("*.json")))

if sidecar_count != 100:
    raise RuntimeError(
        f"sidecar数が100個ではありません: {sidecar_count}"
    )

print(f"教師データ容量: {DATA_PATH.stat().st_size / 1024 / 1024:.2f} MB")
print("sidecar数:", sidecar_count)

# =========================================================
# 4. 出力先準備
# =========================================================

print("\n===== 4. 出力先 =====")

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

BEST_PATH = CHECKPOINT_DIR / "bc-smoke-best.pt"
LATEST_PATH = CHECKPOINT_DIR / "bc-smoke-latest.pt"

print("best:", BEST_PATH)
print("latest:", LATEST_PATH)

# =========================================================
# 最終確認
# =========================================================

print("\n========================================")
print(" 復帰処理完了")
print("========================================")
print("commit一致:", current_commit == EXPECTED_COMMIT)
print("プロジェクト:", PROJECT_DIR.is_dir())
print("node_modules:", node_modules.is_dir())
print("教師データ:", DATA_PATH.is_file())
print("sidecar数:", sidecar_count)

 Tactical-hub Kaggle復帰処理

===== 1. ソースコード =====
既存のリポジトリを使用します。
取得コミット: 1ef9943f0b635f8fbfdf1c2f9a6c91a2e5adc6f3

===== 2. npm依存関係 =====
既存のnode_modulesを使用します。

===== 3. 教師データ =====
教師データ容量: 437.72 MB
sidecar数: 100

===== 4. 出力先 =====
best: /kaggle/working/tactical-hub-output/checkpoints/bc-smoke-best.pt
latest: /kaggle/working/tactical-hub-output/checkpoints/bc-smoke-latest.pt

 復帰処理完了
commit一致: True
プロジェクト: True
node_modules: True
教師データ: True
sidecar数: 100


From https://github.com/YoshimatsuKeisei/Tactical-hub
 * branch            main       -> FETCH_HEAD
HEAD is now at 1ef9943 :feat:testデータの後回し処理,落ちた際のepisode単位のreplay実装,replay indexにより毎先頭走査の削減


In [4]:
from pathlib import Path
import subprocess
import os
import time

PROJECT_DIR = Path("/kaggle/working/Tactical-hub/Tactical-hub")

DATA_PATH = Path(
    "/kaggle/input/datasets/keisabo/imitation-learning-dataset/bc100"
)

CHECKPOINT_DIR = Path(
    "/kaggle/working/tactical-hub-output/checkpoints"
)

BEST_PATH = CHECKPOINT_DIR / "bc-smoke-best.pt"
LATEST_PATH = CHECKPOINT_DIR / "bc-smoke-latest.pt"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("===== 1エポック目：実行前確認 =====")
print("プロジェクト:", PROJECT_DIR.is_dir())
print("教師データ:", DATA_PATH.is_file())
print("best保存先:", BEST_PATH)
print("latest保存先:", LATEST_PATH)

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        "プロジェクトがありません。先に復帰用セルを実行してください。"
    )

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        "教師データbc100がありません。"
    )

# 今回は新規テストなので、以前のsmoke用ファイルだけ削除
for path in [BEST_PATH, LATEST_PATH]:
    if path.exists():
        path.unlink()
        print("古いテストファイルを削除:", path.name)

os.chdir(PROJECT_DIR)

command = [
    "npm",
    "run",
    "rl:bc",
    "--",
    "--data",
    str(DATA_PATH),
    "--epochs",
    "1",
    "--batch-size",
    "8",
    "--learning-rate",
    "0.0001",
    "--checkpoint",
    str(BEST_PATH),
    "--latest-checkpoint",
    str(LATEST_PATH),
    "--train-range",
    "1-1",
    "--validation-range",
    "2-2",
    "--test-range",
    "3-3",
    "--seed",
    "1",
    "--workers",
    "1",
    "--torch-threads",
    "2",
    "--torch-interop-threads",
    "1",
    "--device",
    "cpu",
]

print("\n===== 1エポック目開始 =====")
print(" ".join(command))
print()

started = time.time()

process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

assert process.stdout is not None

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
elapsed = time.time() - started

print("\n===== 1エポック目：最終確認 =====")
print("return code:", return_code)
print(f"経過時間: {elapsed:.1f}秒")
print("best存在:", BEST_PATH.is_file())
print("latest存在:", LATEST_PATH.is_file())

if BEST_PATH.is_file():
    print(
        "best容量:",
        f"{BEST_PATH.stat().st_size / 1024 / 1024:.2f} MB",
    )

if LATEST_PATH.is_file():
    print(
        "latest容量:",
        f"{LATEST_PATH.stat().st_size / 1024 / 1024:.2f} MB",
    )

if return_code == 0 and BEST_PATH.is_file() and LATEST_PATH.is_file():
    print("\n1エポック目の保存成功です。")
else:
    raise RuntimeError(
        "1エポック目の学習またはcheckpoint保存に失敗しました。"
    )

===== 1エポック目：実行前確認 =====
プロジェクト: True
教師データ: True
best保存先: /kaggle/working/tactical-hub-output/checkpoints/bc-smoke-best.pt
latest保存先: /kaggle/working/tactical-hub-output/checkpoints/bc-smoke-latest.pt

===== 1エポック目開始 =====
npm run rl:bc -- --data /kaggle/input/datasets/keisabo/imitation-learning-dataset/bc100 --epochs 1 --batch-size 8 --learning-rate 0.0001 --checkpoint /kaggle/working/tactical-hub-output/checkpoints/bc-smoke-best.pt --latest-checkpoint /kaggle/working/tactical-hub-output/checkpoints/bc-smoke-latest.pt --train-range 1-1 --validation-range 2-2 --test-range 3-3 --seed 1 --workers 1 --torch-threads 2 --torch-interop-threads 1 --device cpu


> rl:bc
> vite-node src/game/cpu/rlBehavioralCloningCli.ts --data /kaggle/input/datasets/keisabo/imitation-learning-dataset/bc100 --epochs 1 --batch-size 8 --learning-rate 0.0001 --checkpoint /kaggle/working/tactical-hub-output/checkpoints/bc-smoke-best.pt --latest-checkpoint /kaggle/working/tactical-hub-output/checkpoints/bc-smoke-la

In [5]:
from pathlib import Path
import subprocess
import os
import time

PROJECT_DIR = Path("/kaggle/working/Tactical-hub/Tactical-hub")

DATA_PATH = Path(
    "/kaggle/input/datasets/keisabo/imitation-learning-dataset/bc100"
)

CHECKPOINT_DIR = Path(
    "/kaggle/working/tactical-hub-output/checkpoints"
)

BEST_PATH = CHECKPOINT_DIR / "bc-smoke-best.pt"
LATEST_PATH = CHECKPOINT_DIR / "bc-smoke-latest.pt"

print("===== 再開前確認 =====")
print("best存在:", BEST_PATH.is_file())
print("latest存在:", LATEST_PATH.is_file())

if not BEST_PATH.is_file():
    raise FileNotFoundError(
        "best checkpointがありません。1エポック目を先に実行してください。"
    )

if not LATEST_PATH.is_file():
    raise FileNotFoundError(
        "latest checkpointがありません。1エポック目を先に実行してください。"
    )

os.chdir(PROJECT_DIR)

command = [
    "npm",
    "run",
    "rl:bc",
    "--",
    "--data",
    str(DATA_PATH),
    "--epochs",
    "2",
    "--batch-size",
    "8",
    "--learning-rate",
    "0.0001",
    "--checkpoint",
    str(BEST_PATH),
    "--latest-checkpoint",
    str(LATEST_PATH),
    "--resume",
    str(LATEST_PATH),
    "--train-range",
    "1-1",
    "--validation-range",
    "2-2",
    "--test-range",
    "3-3",
    "--seed",
    "1",
    "--workers",
    "1",
    "--torch-threads",
    "2",
    "--torch-interop-threads",
    "1",
    "--device",
    "cpu",
]

print("\n===== latestから再開 =====")
print(" ".join(command))
print()

started = time.time()

process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

assert process.stdout is not None

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
elapsed = time.time() - started

print("\n===== 再開テスト：最終確認 =====")
print("return code:", return_code)
print(f"経過時間: {elapsed:.1f}秒")
print("best存在:", BEST_PATH.is_file())
print("latest存在:", LATEST_PATH.is_file())

if return_code == 0:
    print("\n再開して総2エポックへ到達しました。")
else:
    raise RuntimeError(
        "latest checkpointからの再開に失敗しました。"
    )

===== 再開前確認 =====
best存在: True
latest存在: True

===== latestから再開 =====
npm run rl:bc -- --data /kaggle/input/datasets/keisabo/imitation-learning-dataset/bc100 --epochs 2 --batch-size 8 --learning-rate 0.0001 --checkpoint /kaggle/working/tactical-hub-output/checkpoints/bc-smoke-best.pt --latest-checkpoint /kaggle/working/tactical-hub-output/checkpoints/bc-smoke-latest.pt --resume /kaggle/working/tactical-hub-output/checkpoints/bc-smoke-latest.pt --train-range 1-1 --validation-range 2-2 --test-range 3-3 --seed 1 --workers 1 --torch-threads 2 --torch-interop-threads 1 --device cpu


> rl:bc
> vite-node src/game/cpu/rlBehavioralCloningCli.ts --data /kaggle/input/datasets/keisabo/imitation-learning-dataset/bc100 --epochs 2 --batch-size 8 --learning-rate 0.0001 --checkpoint /kaggle/working/tactical-hub-output/checkpoints/bc-smoke-best.pt --latest-checkpoint /kaggle/working/tactical-hub-output/checkpoints/bc-smoke-latest.pt --resume /kaggle/working/tactical-hub-output/checkpoints/bc-smoke-late

In [2]:
import torch
import subprocess

print("PyTorch:", torch.__version__)
print("CUDA利用可能:", torch.cuda.is_available())
print("GPU数:", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError("CUDAが利用できません。")

for index in range(torch.cuda.device_count()):
    print(
        f"GPU {index}:",
        torch.cuda.get_device_name(index),
        "capability:",
        torch.cuda.get_device_capability(index),
    )

# 実際にCUDA計算できるか確認
x = torch.randn(1000, 1000, device="cuda")
y = x @ x

torch.cuda.synchronize()

print("CUDA計算テスト: 成功")
print("結果device:", y.device)

print("\n===== nvidia-smi =====")
subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=index,name,memory.total,memory.free",
        "--format=csv,noheader",
    ],
    check=True,
)

PyTorch: 2.10.0+cu128
CUDA利用可能: True
GPU数: 2
GPU 0: Tesla T4 capability: (7, 5)
GPU 1: Tesla T4 capability: (7, 5)
CUDA計算テスト: 成功
結果device: cuda:0

===== nvidia-smi =====
0, Tesla T4, 15360 MiB, 14775 MiB
1, Tesla T4, 15360 MiB, 14909 MiB


CompletedProcess(args=['nvidia-smi', '--query-gpu=index,name,memory.total,memory.free', '--format=csv,noheader'], returncode=0)

In [3]:
from pathlib import Path
import subprocess
import os
import time

PROJECT_DIR = Path("/kaggle/working/Tactical-hub/Tactical-hub")
DATA_PATH = Path(
    "/kaggle/input/datasets/keisabo/imitation-learning-dataset/bc100"
)

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        "プロジェクトがありません。先に復帰用セルを実行してください。"
    )

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"教師データがありません: {DATA_PATH}"
    )

os.chdir(PROJECT_DIR)

batch_sizes = [8, 16, 64, 128, 256]
results = []

for batch_size in batch_sizes:
    print("\n" + "=" * 70)
    print(f"batch size {batch_size} のGPU速度測定")
    print("=" * 70)

    command = [
        "npm",
        "run",
        "rl:bc-profile",
        "--",
        "--data",
        str(DATA_PATH),
        "--samples",
        "4096",
        "--warmup-samples",
        "256",
        "--batch-size",
        str(batch_size),
        "--workers",
        "1",
        "--profile-episodes",
        "4",
        "--device",
        "cuda",
    ]

    print(" ".join(command))
    print()

    started = time.time()

    completed = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    elapsed = time.time() - started

    print(completed.stdout)

    results.append({
        "batch_size": batch_size,
        "return_code": completed.returncode,
        "elapsed_seconds": elapsed,
    })

    if completed.returncode != 0:
        print(
            f"batch size {batch_size} は失敗しました。"
            "GPUメモリ不足または別のエラーの可能性があります。"
        )

print("\n" + "=" * 70)
print("測定結果一覧")
print("=" * 70)

for result in results:
    print(
        f"batch size={result['batch_size']:>3} | "
        f"return code={result['return_code']} | "
        f"セル実測={result['elapsed_seconds']:.2f}秒"
    )


batch size 8 のGPU速度測定
npm run rl:bc-profile -- --data /kaggle/input/datasets/keisabo/imitation-learning-dataset/bc100 --samples 4096 --warmup-samples 256 --batch-size 8 --workers 1 --profile-episodes 4 --device cuda


> rl:bc-profile
> vite-node src/game/cpu/rlBcProfilerCli.ts --data /kaggle/input/datasets/keisabo/imitation-learning-dataset/bc100 --samples 4096 --warmup-samples 256 --batch-size 8 --workers 1 --profile-episodes 4 --device cuda

The CJS build of Vite's Node API is deprecated. See https://vite.dev/guide/troubleshooting.html#vite-cjs-node-api-deprecated for more details.
[RL Device] requested device=cuda | selected device=cuda | PyTorch=2.10.0+cu128 | cuda.is_available=True | GPU=Tesla T4 | PyTorch CUDA=12.8
{
  "profileEpisodeCount": 4,
  "episodeNumbers": [
    1,
    2,
    3,
    4
  ],
  "samplesPerEpisode": [
    {
      "episodeNumber": 1,
      "warmupSamples": 64,
      "measuredSamples": 1024
    },
    {
      "episodeNumber": 2,
      "warmupSamples": 64,
    